# M5 Kaggle upload test — CPU only

This small test checks whether Kaggle preserves one uploaded archive's exact bytes,
including a gzip fixture and a nested ZIP with a synthetic SQLite file.

1. Create a **private dataset** named `m5-source-canary` and upload only
   **`m5-source-canary.payload`**, keeping that filename and extension.
2. Import this notebook into a **private Kaggle notebook**. Set **Accelerator: None**
   and **Internet: Off**. Attach the new canary dataset using Add Input.
3. If convenient, also attach the original `m5-kaggle-preflight` dataset to collect
   its file inventory. This is optional; the upload test works with just the canary.
4. Select **Run All**. Download the result ZIP from the last cell, stop the session,
   and put the ZIP and the executed notebook in your local `notebooks/run/` folder.

Keep the payload unchanged. If Kaggle refuses the upload, return its error message.
All fixture data are synthetic. This notebook installs no packages, downloads no
weights and performs no language-model generations. A passing result verifies the
tiny transport candidate; full-source preparation and model acceptance remain open.


In [1]:
import hashlib, json, os
from pathlib import Path

CANARY_PROGRAM = '"""CPU-only, dependency-free diagnosis of Kaggle archive transport.\n\nAll fixtures are synthetic. This does not validate the study or load a model.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nfrom contextlib import closing\nfrom datetime import datetime, timezone\nimport gzip\nimport hashlib\nimport io\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport platform\nimport sqlite3\nimport stat\nimport time\nimport uuid\nimport zipfile\n\n\nBASELINE_COMMIT = "ca46bcd55486b38bb453f5294e01f943bc7f7752"\nGZIP_PATH = "tests/fixtures/capacity_report.json.gz"\nNESTED_PATH = "tests/fixtures/scripted_lifecycle_run.zip"\nINVENTORY_LIMIT = 2000\nPAYLOAD_LIMIT = 1024 * 1024\n\n\ndef sha256(data: bytes) -> str:\n    return hashlib.sha256(data).hexdigest()\n\n\ndef write_json(path: Path, data: object) -> None:\n    path.write_text(json.dumps(data, indent=2, ensure_ascii=False) + "\\n", encoding="utf-8")\n\n\ndef visible_gpu_devices() -> list[str]:\n    """Inspect driver entries without importing Torch or initializing CUDA."""\n    location = Path("/proc/driver/nvidia/gpus")\n    return sorted(p.name for p in location.iterdir()) if location.is_dir() else []\n\n\ndef inventory(root: Path, limit: int = INVENTORY_LIMIT) -> dict:\n    """Bound the scan and never follow an attached symlink."""\n    if not root.is_dir():\n        raise ValueError(f"input_root_missing: {root}")\n    entries, errors, pending = [], [], [root]\n    truncated = False\n    while pending:\n        directory = pending.pop()\n        try:\n            with os.scandir(directory) as children:\n                for child in children:\n                    if len(entries) >= limit:\n                        truncated = True\n                        break\n                    path = Path(child.path)\n                    item = {"path": path.relative_to(root).as_posix()}\n                    try:\n                        info = child.stat(follow_symlinks=False)\n                        item["bytes"] = info.st_size\n                        if child.is_symlink():\n                            item["kind"] = "symlink"\n                        elif child.is_dir(follow_symlinks=False):\n                            item["kind"] = "directory"\n                            pending.append(path)\n                        elif child.is_file(follow_symlinks=False):\n                            item["kind"] = "file"\n                        else:\n                            item["kind"] = "other"\n                    except OSError as exc:\n                        item["kind"] = "error"\n                        errors.append({"path": item["path"], "reason": str(exc)})\n                    entries.append(item)\n        except OSError as exc:\n            errors.append({"path": directory.relative_to(root).as_posix(), "reason": str(exc)})\n        if truncated:\n            break\n    return {"root": str(root), "limit": limit, "truncated": truncated, "errors": errors,\n            "entries": sorted(entries, key=lambda item: item["path"])}\n\n\ndef legacy_source_observations(root: Path, listing: dict) -> dict:\n    """R1 observations are separate from admission of the synthetic R2 payload."""\n    sources = []\n    for item in listing["entries"]:\n        path = root / item["path"]\n        if item["kind"] != "file" or path.name != "SOURCE_COMMIT.txt" or item["bytes"] > 128:\n            continue\n        marker = path.read_bytes().decode("utf-8", errors="replace").strip()\n        if marker != BASELINE_COMMIT:\n            continue\n        source = path.parent\n        observed = {"root": str(source), "baseline_marker": marker, "paths": {}}\n        for name in [GZIP_PATH, GZIP_PATH[:-3], NESTED_PATH, NESTED_PATH[:-4]]:\n            candidate = source / name\n            # Check every component so a symlinked fixtures directory is not followed.\n            current = source\n            linked = False\n            for part in PurePosixPath(name).parts:\n                current = current / part\n                if current.is_symlink():\n                    linked = True\n                    break\n            kind = "symlink" if linked else (\n                "file" if candidate.is_file() else "directory" if candidate.is_dir() else "missing")\n            observed["paths"][name] = {"kind": kind}\n            if kind == "file":\n                observed["paths"][name]["bytes"] = candidate.stat().st_size\n        sources.append(observed)\n    return {"status": "observed" if sources else "not_observed", "sources": sources,\n            "complete_inventory": not listing["truncated"] and not listing["errors"],\n            "interpretation": "Path observations only; no automatic recompression or source repair."}\n\n\ndef verify_zip(data: bytes, expected: dict[str, dict]) -> dict[str, bytes]:\n    """Verify bounded member bytes without extracting an untrusted tree."""\n    if len(data) > PAYLOAD_LIMIT or sum(item["bytes"] for item in expected.values()) > PAYLOAD_LIMIT:\n        raise ValueError("archive_too_large: this is a tiny synthetic canary")\n    with zipfile.ZipFile(io.BytesIO(data)) as archive:\n        infos = archive.infolist()\n        names = [item.filename for item in infos]\n        if len(names) != len(set(names)):\n            raise ValueError("duplicate_archive_member")\n        for item in infos:\n            name = item.filename\n            path = PurePosixPath(name)\n            if (not name or "\\\\" in name or ":" in name or name.startswith("/")\n                    or any(part in {"", ".", ".."} for part in name.split("/"))\n                    or path.is_absolute()):\n                raise ValueError(f"unsafe_archive_path: {name}")\n            if stat.S_ISLNK(item.external_attr >> 16):\n                raise ValueError(f"archive_symlink: {name}")\n        if set(names) != set(expected):\n            raise ValueError(f"archive_member_set: missing={sorted(set(expected)-set(names))}; "\n                             f"unexpected={sorted(set(names)-set(expected))}")\n        members = {}\n        for item in infos:\n            wanted = expected[item.filename]\n            if item.file_size != wanted["bytes"]:\n                raise ValueError(f"archive_member_size: {item.filename}")\n            content = archive.read(item)\n            if sha256(content) != wanted["sha256"]:\n                raise ValueError(f"archive_member_hash: {item.filename}")\n            members[item.filename] = content\n        return members\n\n\ndef verify_payload(path: Path, expected: dict) -> dict[str, bytes]:\n    if path.is_symlink() or not path.is_file():\n        raise ValueError("payload_not_regular_file")\n    if expected["archive_bytes"] > PAYLOAD_LIMIT or path.stat().st_size != expected["archive_bytes"]:\n        raise ValueError("payload_size_mismatch")\n    data = path.read_bytes()\n    if sha256(data) != expected["archive_sha256"]:\n        raise ValueError("payload_hash_mismatch")\n    return verify_zip(data, expected["members"])\n\n\ndef check_sqlite(path: Path) -> dict:\n    with closing(sqlite3.connect(path.resolve().as_uri() + "?mode=ro&immutable=1", uri=True)) as db:\n        integrity = db.execute("PRAGMA integrity_check").fetchone()[0]\n        rows = db.execute("SELECT value FROM probe").fetchall()\n        try:\n            db.execute("INSERT INTO probe VALUES (8)")\n        except sqlite3.OperationalError as exc:\n            if "readonly" not in str(exc).lower():\n                raise\n        else:\n            raise ValueError("sqlite_read_only_not_enforced")\n    if integrity != "ok" or rows != [(7,)]:\n        raise ValueError(f"sqlite_contents: integrity={integrity}, rows={rows}")\n    return {"integrity": integrity, "rows": [[7]], "read_only_enforced": True}\n\n\ndef filesystem_probe(run: Path) -> dict:\n    original, backup = run / "filesystem.sqlite", run / "filesystem-backup.sqlite"\n    with closing(sqlite3.connect(original, timeout=1)) as db:\n        journal = db.execute("PRAGMA journal_mode=WAL").fetchone()[0]\n        db.execute("PRAGMA synchronous=FULL")\n        synchronous = db.execute("PRAGMA synchronous").fetchone()[0]\n        db.execute("CREATE TABLE probe (value INTEGER NOT NULL)")\n        db.execute("INSERT INTO probe VALUES (7)")\n        db.commit()\n        with closing(sqlite3.connect(backup, timeout=1)) as destination:\n            db.backup(destination)\n    if journal != "wal" or synchronous != 2:\n        raise ValueError(f"sqlite_settings: journal={journal}, synchronous={synchronous}")\n    return {"journal_mode": journal, "synchronous": synchronous, "backup": check_sqlite(backup),\n            "scope": "Small synthetic host diagnostic; actual worker recovery remains untested."}\n\n\ndef export_results(run: Path) -> tuple[Path, str]:\n    names = ["canary-report.json", "summary.json", "input-inventory.json", "expected-manifest.json",\n             "transport.sqlite", "filesystem.sqlite", "filesystem-backup.sqlite"]\n    retained = [name for name in names if (run / name).is_file()]\n    temporary, final = run / "canary-results.zip.partial", run / "canary-results.zip"\n    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED) as archive:\n        for name in retained:\n            archive.write(run / name, name)\n    with zipfile.ZipFile(temporary) as archive:\n        if archive.testzip() is not None or set(archive.namelist()) != set(retained):\n            raise ValueError("export_integrity_failed")\n    temporary.replace(final)\n    digest = sha256(final.read_bytes())\n    (run / "canary-results.sha256").write_text(f"{digest}  {final.name}\\n", encoding="utf-8")\n    return final, digest\n\n\ndef run_canary(input_root: Path, output_root: Path, expected: dict) -> dict:\n    input_root, output_root = Path(input_root).resolve(), Path(output_root).resolve()\n    label = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + "-" + uuid.uuid4().hex[:8]\n    run = output_root / ("recovery-m5-upload-canary-" + label)\n    run.mkdir(parents=True, exist_ok=False)\n    started = time.monotonic()\n    report = {"schema_version": 1, "kind": "synthetic-kaggle-transport-canary",\n              "started_utc": datetime.now(timezone.utc).isoformat(), "output_directory": str(run),\n              "expected_archive_sha256": expected["archive_sha256"], "stages": {},\n              "legacy_source": {"status": "not_observed"}}\n    write_json(run / "expected-manifest.json", expected)\n\n    def stage(name, action):\n        entry = {"status": "not_run", "reason": None}\n        report["stages"][name] = entry\n        before = time.monotonic()\n        try:\n            if before - started >= 60:\n                raise TimeoutError("canary_deadline_reached")\n            value = action()\n        except (Exception, KeyboardInterrupt) as exc:\n            entry.update(status="failed", reason=f"{type(exc).__name__}: {exc}")\n            value = None\n        else:\n            entry["status"] = "passed"\n        entry["elapsed_seconds"] = round(time.monotonic() - before, 6)\n        write_json(run / "canary-report.json", report)\n        return value\n\n    def host_check():\n        gpus = visible_gpu_devices()\n        report["host"] = {"python": platform.python_version(), "os": platform.platform(),\n                          "sqlite": sqlite3.sqlite_version, "input_root": str(input_root),\n                          "visible_driver_gpu_entries": gpus,\n                          "gpu_probe": "Driver directory only; no CUDA initialization or billing claim."}\n        if gpus:\n            raise ValueError("gpu_visible: set Kaggle accelerator to None and rerun on CPU")\n        return True\n\n    def input_check():\n        listing = inventory(input_root)\n        write_json(run / "input-inventory.json", listing)\n        report["legacy_source"] = legacy_source_observations(input_root, listing)\n        if listing["truncated"] or listing["errors"]:\n            raise ValueError("input_inventory_incomplete: remove unrelated inputs or report the scan errors")\n        return listing\n\n    try:\n        host_ready = stage("cpu_host", host_check)\n        listing = stage("input_inventory", input_check)\n        members = None\n        if host_ready and listing is not None:\n            def payload_check():\n                candidates = [input_root / item["path"] for item in listing["entries"]\n                              if item["kind"] == "file"\n                              and Path(item["path"]).name == expected["payload_filename"]]\n                if not candidates:\n                    raise ValueError(f"payload_missing: expected {expected[\'payload_filename\']}; "\n                                     "see input-inventory.json for transformed or absent input")\n                if len(candidates) != 1:\n                    raise ValueError("payload_ambiguous: attach exactly one copy of the canary dataset")\n                report["selected_payload"] = str(candidates[0])\n                return verify_payload(candidates[0], expected)\n            members = stage("transport_bytes", payload_check)\n        else:\n            report["stages"]["transport_bytes"] = {"status": "blocked", "reason": "cpu_host/input_inventory"}\n\n        if members is not None:\n            def compressed_check():\n                with gzip.GzipFile(fileobj=io.BytesIO(members[GZIP_PATH])) as stream:\n                    raw = stream.read(expected["gzip_decoded"]["bytes"] + 1)\n                if len(raw) != expected["gzip_decoded"]["bytes"] or sha256(raw) != expected["gzip_decoded"]["sha256"]:\n                    raise ValueError("gzip_decoded_bytes_mismatch")\n                nested = verify_zip(members[NESTED_PATH], expected["nested_members"])\n                (run / "transport.sqlite").write_bytes(nested["worker.sqlite"])\n                report["compressed_contents"] = {"gzip_bytes_preserved": True, "nested_zip_bytes_preserved": True,\n                                                  "nested_sqlite": check_sqlite(run / "transport.sqlite")}\n                return True\n            stage("compressed_contents", compressed_check)\n        else:\n            report["stages"]["compressed_contents"] = {"status": "blocked", "reason": "transport_bytes"}\n        probe = stage("filesystem_backup", lambda: filesystem_probe(run))\n        if probe is not None:\n            report["filesystem"] = probe\n    finally:\n        required = ["cpu_host", "input_inventory", "transport_bytes", "compressed_contents", "filesystem_backup"]\n        for name in required:\n            report["stages"].setdefault(name, {"status": "not_run", "reason": "earlier interruption"})\n        failures = [f"{name}: {entry[\'reason\']}" for name, entry in report["stages"].items() if entry["status"] == "failed"]\n        passed = all(report["stages"][name]["status"] == "passed" for name in required)\n        summary = {"canary_checks_passed": passed, "root_blocker": failures[0] if failures else None,\n                   "transport": "raw_payload", "payload_filename": expected["payload_filename"],\n                   "model_generations": 0, "full_source_preparation": None, "model_smoke": None,\n                   "legacy_source_status": report["legacy_source"]["status"],\n                   "observed_wall_seconds": round(time.monotonic() - started, 6),\n                   "provider_assignment_utc": None, "provider_release_utc": None,\n                   "next_action": "Download this ZIP, stop the session, and return it for transport review."\n                                  if passed else "Download this ZIP and return the failure report; keep the original payload unchanged."}\n        report["summary"] = summary\n        write_json(run / "canary-report.json", report)\n        write_json(run / "summary.json", summary)\n        try:\n            result_zip, digest = export_results(run)\n        except Exception as exc:\n            report["export_error"] = f"{type(exc).__name__}: {exc}"\n            write_json(run / "canary-report.json", report)\n            result_zip, digest = None, None\n    return {"report": report, "summary": summary, "run_directory": str(run),\n            "zip_path": str(result_zip) if result_zip else None, "zip_sha256": digest}\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--input-root", type=Path, default=Path("/kaggle/input"))\n    parser.add_argument("--output-root", type=Path, default=Path("/kaggle/working"))\n    parser.add_argument("--manifest", type=Path, required=True)\n    args = parser.parse_args()\n    result = run_canary(args.input_root, args.output_root, json.loads(args.manifest.read_text(encoding="utf-8")))\n    print(json.dumps({key: value for key, value in result.items() if key != "report"}, indent=2))\n    return 0 if result["summary"]["canary_checks_passed"] and result["zip_path"] else 1\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
EXPECTED = json.loads('{"archive_bytes": 1173, "archive_sha256": "887d5d832546eb9e125812f35493a9a3d22b5e33d02849304e8dd608c3ae6bb2", "build_environment": {"python": "3.11.9", "sqlite": "3.45.1"}, "build_source_commit": "c20ebea1382e01a82de08d023a2e04f6085409d2", "gzip_decoded": {"bytes": 104, "sha256": "182aeb8314e7017361a2ed48cdc2452129476ea258fc90226ec6ec4b9093aae0"}, "helper_sha256": "d08a2fa62ab76addfab8c0dd74d9c0bd4f367a62ae6a9df62fb41fee5124f87a", "kind": "synthetic-source-transport-canary", "members": {"SOURCE_COMMIT.txt": {"bytes": 33, "sha256": "1ae5d9e7e779bf34691da73f85c51fdd4eba20270775a32a1ea59cb7928d48a7"}, "canary.json": {"bytes": 197, "sha256": "16e2a758219afcfbd41c97cdce641de0a9a430549032ac6895ae43b0e3610276"}, "tests/fixtures/capacity_report.json.gz": {"bytes": 122, "sha256": "5d5430101035d70f0724787f531e49150fc5773c6417cd07d4b7b4a7d2305bfe"}, "tests/fixtures/scripted_lifecycle_run.zip": {"bytes": 428, "sha256": "afad647b13a884b79872622e7c4435377b4cf97d592f0281a0e3e7dacaff6b16"}}, "nested_members": {"report.json": {"bytes": 48, "sha256": "e5efff472193ccdff4f45483882a723b20ea48185c960363de033b8288cafe3f"}, "worker.sqlite": {"bytes": 8192, "sha256": "5e516b46e32992539d7e29ded5a7945fa5eddc1c7160ca22ef3301f0a301791f"}}, "payload_filename": "m5-source-canary.payload", "schema_version": 1, "source_baseline_commit": "ca46bcd55486b38bb453f5294e01f943bc7f7752", "transport": "raw_payload"}')
assert hashlib.sha256(CANARY_PROGRAM.encode('utf-8')).hexdigest() == EXPECTED['helper_sha256']
canary_namespace = {'__name__': 'recovery_m5_transport_canary'}
exec(compile(CANARY_PROGRAM, 'm5_kaggle_upload_canary.py', 'exec'), canary_namespace)


## Run the bounded input and backup checks


In [2]:
# Defaults need no edits on Kaggle. Overrides are for controlled local validation.
input_root = Path(os.environ.get('M5_CANARY_INPUT_ROOT', '/kaggle/input'))
output_root = Path(os.environ.get('M5_CANARY_OUTPUT_ROOT', '/kaggle/working'))
CANARY_RESULT = canary_namespace['run_canary'](input_root, output_root, EXPECTED)
print(json.dumps(CANARY_RESULT['summary'], indent=2))


{
  "canary_checks_passed": true,
  "root_blocker": null,
  "transport": "raw_payload",
  "payload_filename": "m5-source-canary.payload",
  "model_generations": 0,
  "full_source_preparation": null,
  "model_smoke": null,
  "legacy_source_status": "not_observed",
  "observed_wall_seconds": 0.063,
  "provider_assignment_utc": null,
  "provider_release_utc": null,
  "next_action": "Download this ZIP, stop the session, and return it for transport review."
}


## Download the evidence
Both success and handled failure produce diagnostics.


In [3]:
from IPython.display import FileLink, display

if CANARY_RESULT['zip_path']:
    print('Download this result ZIP, then stop the Kaggle session:')
    print('SHA-256:', CANARY_RESULT['zip_sha256'])
    zip_file = Path(CANARY_RESULT['zip_path'])
    display(FileLink(os.path.relpath(zip_file, Path.cwd())))
else:
    print('Export failed. Retain the JSON files in:', CANARY_RESULT['run_directory'])
    print(CANARY_RESULT['report'].get('export_error'))
print('No model calls were made. Return the results before running the original preflight again.')


Download this result ZIP, then stop the Kaggle session:
SHA-256: 473ca98b001123d14fc0e08bef01a0afc24f17b80aa65b21d808cee367228ab2


C:\Users\EndritHasani\AppData\Local\Temp\m5-canary-selector-check-z3w2faby\valid\working\recovery-m5-upload-canary-20261008T155335590499Z-f7c3c323\canary-results.zip

No model calls were made. Return the results before running the original preflight again.
